In [ ]:
import os
import zipfile

import kaggle
import pandas as pd
import numpy as np

In [ ]:
# Download the original Kaggle dataset.
# The result is expected to be a ZIP archive.

!kaggle datasets download -d syednaveed05/logistics-fleet-data

In [ ]:
# Display all files in the current working directory.

for file in os.listdir():
    print(file)

In [ ]:
zip_path = "logistics-fleet-data.zip"

# Open the ZIP without extracting it yet.
with zipfile.ZipFile(zip_path, "r") as zip_ref:

    # Show every file stored inside the archive.
    files_inside_zip = zip_ref.namelist()

files_inside_zip

In [ ]:
# Keep raw files separate from cleaned files.
# This allows the original source data to remain untouched.

raw_path = "data/raw"

os.makedirs(raw_path, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_ref:

    # Extract all original files into data/raw.
    zip_ref.extractall(raw_path)

In [ ]:
# Expected files based on the ZIP inspection.

expected_files = [
    "DimensionTables.xlsx",
    "fCosts.xlsx",
    "fFreight.csv"
]

# Verify that every expected file actually exists.

for file in expected_files:

    file_path = os.path.join(raw_path, file)

    print(
        f"{file:<25} -> "
        f"{'FOUND' if os.path.exists(file_path) else 'MISSING'}"
    )

In [ ]:
dimension_path = os.path.join(
    raw_path,
    "DimensionTables.xlsx"
)

# sheet_name=None tells Pandas to read every sheet.
df_dim = pd.read_excel(
    dimension_path,
    sheet_name=None
)

In [ ]:
# First question:
# What Python object did Pandas give us?

type(df_dim)

In [ ]:
# Dictionary keys represent the Excel sheet names.

list(df_dim.keys())

In [ ]:
# Inspect the object stored under every dictionary key.

for sheet_name, data in df_dim.items():

    print("=" * 70)
    print("SHEET:", sheet_name)
    print("OBJECT TYPE:", type(data))
    print("=" * 70)

In [ ]:
# Explicitly verify that every workbook sheet was loaded as a DataFrame.

for sheet_name, data in df_dim.items():

    print(
        f"{sheet_name:<15} "
        f"-> DataFrame: {isinstance(data, pd.DataFrame)}"
    )

In [ ]:
drivers_raw = df_dim["Drivers"].copy()

print("Shape:", drivers_raw.shape)
print("\nColumns:")
print(drivers_raw.columns.tolist())

print("\nData types:")
print(drivers_raw.dtypes)

print("\nMissing values:")
print(drivers_raw.isna().sum())

print("\nDuplicate rows:")
print(drivers_raw.duplicated().sum())

display(drivers_raw.head())

In [ ]:
# Inspect the number of unique values in every column.

drivers_raw.nunique()

In [ ]:
# Specifically investigate Driver ID.

print("Number of rows:", len(drivers_raw))
print("Unique Driver IDs:", drivers_raw["Driver ID"].nunique())

In [ ]:
print(
    "Missing Driver IDs:",
    drivers_raw["Driver ID"].isna().sum()
)

print(
    "Duplicate Driver IDs:",
    drivers_raw["Driver ID"].duplicated().sum()
)

print(
    "Driver ID unique:",
    drivers_raw["Driver ID"].is_unique
)

In [ ]:
drivers_raw[
    drivers_raw["Driver"] == "No Driver"
]

In [ ]:
vehicles_raw = df_dim["Vehicles"].copy()

print("Shape:", vehicles_raw.shape)
print("\nColumns:")
print(vehicles_raw.columns.tolist())

print("\nData types:")
print(vehicles_raw.dtypes)

print("\nMissing values:")
print(vehicles_raw.isna().sum())

print("\nDuplicate rows:")
print(vehicles_raw.duplicated().sum())

display(vehicles_raw.head())

In [ ]:
vehicles_raw.nunique()

In [ ]:
print("Rows:", len(vehicles_raw))
print(
    "Unique Truck IDs:",
    vehicles_raw["Truck ID"].nunique()
)

In [ ]:
print(
    "Missing Truck IDs:",
    vehicles_raw["Truck ID"].isna().sum()
)

print(
    "Duplicate Truck IDs:",
    vehicles_raw["Truck ID"].duplicated().sum()
)

print(
    "Truck ID unique:",
    vehicles_raw["Truck ID"].is_unique
)

In [ ]:
customers_raw = df_dim["Customers"].copy()

print("Shape:", customers_raw.shape)
print("\nColumns:")
print(customers_raw.columns.tolist())

print("\nData types:")
print(customers_raw.dtypes)

print("\nMissing values:")
print(customers_raw.isna().sum())

print("\nDuplicate rows:")
print(customers_raw.duplicated().sum())

display(customers_raw.head())

In [ ]:
customers_raw.nunique()

In [ ]:
print("Rows:", len(customers_raw))

print(
    "Unique Customer IDs:",
    customers_raw["Customer ID"].nunique()
)

In [ ]:
print(
    "Missing Customer IDs:",
    customers_raw["Customer ID"].isna().sum()
)

print(
    "Duplicate Customer IDs:",
    customers_raw["Customer ID"].duplicated().sum()
)

print(
    "Customer ID unique:",
    customers_raw["Customer ID"].is_unique
)

In [ ]:
costs_path = os.path.join(
    raw_path,
    "fCosts.xlsx"
)

costs_raw = pd.read_excel(costs_path)

print("Python object:", type(costs_raw))
print("Shape:", costs_raw.shape)

In [ ]:
print("Columns:")
print(costs_raw.columns.tolist())

print("\nData types:")
print(costs_raw.dtypes)

print("\nFirst rows:")
display(costs_raw.head(20))

In [ ]:
display(
    costs_raw.iloc[:, 0].head(50)
)

In [ ]:
# Search the first column for rows containing "Date".
# A real repeated header will contain "Date" instead of a date value.

date_header_mask = (
    costs_raw.iloc[:, 0]
    .astype("string")
    .str.strip()
    .eq("Date")
)

header_locations = costs_raw.index[date_header_mask]

print("Header row locations:")
print(header_locations.tolist())

print(
    "\nNumber of header rows found:",
    date_header_mask.sum()
)

In [ ]:
# Inspect rows where the first column cannot be interpreted as a date.

parsed_dates = pd.to_datetime(
    costs_raw.iloc[:, 0],
    errors="coerce"
)

non_date_rows = costs_raw[
    parsed_dates.isna()
]

display(non_date_rows.head(30))

In [ ]:
# Create a parsed date temporarily.

costs_working = costs_raw.copy()

costs_working["parsed_date"] = pd.to_datetime(
    costs_working.iloc[:, 0],
    errors="coerce"
)

# Keep rows that contain a valid date.

costs_working = costs_working[
    costs_working["parsed_date"].notna()
].copy()

print("Rows after structural filtering:", len(costs_working))

display(costs_working.head())

In [ ]:
# Remove the temporary helper column.

costs_working = costs_working.drop(
    columns=["parsed_date"]
)

# Keep the actual eight business columns.
# The raw source contains the fields required for the Costs table.

costs = costs_working.iloc[:, :8].copy()

# Assign clear column names based on the observed source structure.

costs.columns = [
    "date",
    "truck_id",
    "driver_id",
    "km_traveled",
    "liters",
    "fuel",
    "maintenance",
    "fixed_costs"
]

costs = costs.reset_index(drop=True)

display(costs.head())

In [ ]:
costs.dtypes

In [ ]:
costs.info()

In [ ]:
costs["date"] = pd.to_datetime(
    costs["date"],
    errors="coerce"
)

In [ ]:
costs["truck_id"] = pd.to_numeric(
    costs["truck_id"],
    errors="coerce"
).astype("Int64")

costs["driver_id"] = pd.to_numeric(
    costs["driver_id"],
    errors="coerce"
).astype("Int64")

In [ ]:
cost_numeric_columns = [
    "km_traveled",
    "liters",
    "fuel",
    "maintenance",
    "fixed_costs"
]

for column in cost_numeric_columns:

    costs[column] = pd.to_numeric(
        costs[column],
        errors="coerce"
    )

costs.dtypes

In [ ]:
costs.isna().sum()

In [ ]:
print(
    "Duplicate complete rows:",
    costs.duplicated().sum()
)

In [ ]:
invalid_distance = costs[
    costs["km_traveled"] < 0
]

display(invalid_distance)

In [ ]:
invalid_liters = costs[
    costs["liters"] < 0
]

display(invalid_liters)

In [ ]:
invalid_costs = costs[
    (costs["fuel"] < 0) |
    (costs["maintenance"] < 0) |
    (costs["fixed_costs"] < 0)
]

display(invalid_costs)

In [ ]:
zero_activity = costs[
    (costs["km_traveled"] == 0) |
    (costs["liters"] == 0)
]

display(zero_activity.head())

In [ ]:
freight_path = os.path.join(
    raw_path,
    "fFreight.csv"
)

freight_raw = pd.read_csv(
    freight_path
)

print("Python object:", type(freight_raw))
print("Shape:", freight_raw.shape)

In [ ]:
print("Columns:")
print(freight_raw.columns.tolist())

print("\nData types:")
print(freight_raw.dtypes)

print("\nMissing values:")
print(freight_raw.isna().sum())

print("\nDuplicate rows:")
print(freight_raw.duplicated().sum())

display(freight_raw.head())

In [ ]:
freight = freight_raw.copy()

freight.columns = (
    freight.columns
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r"[()]", "", regex=True)
    .str.replace(r"[\s\-\/]+", "_", regex=True)
    .str.replace(r"_+", "_", regex=True)
    .str.strip("_")
)

freight.columns.tolist()

In [ ]:
for column in [
    "net_revenue",
    "weight_kg",
    "weight_cubic",
    "goods_value"
]:

    print(f"\n{column}")
    display(freight[column].head(10))

In [ ]:
decimal_columns = [
    "net_revenue",
    "weight_kg",
    "weight_cubic"
]

for column in decimal_columns:

    freight[column] = (
        freight[column]
        .astype("string")
        .str.strip()
        .str.replace(",", ".", regex=False)
    )

In [ ]:
freight["goods_value"] = (
    freight["goods_value"]
    .astype("string")
    .str.strip()
    .str.replace(",", "", regex=False)
)

In [ ]:
freight["date"] = pd.to_datetime(
    freight["date"],
    errors="coerce"
)

In [ ]:
freight["customer_id"] = pd.to_numeric(
    freight["customer_id"],
    errors="coerce"
).astype("Int64")

freight["truck_id"] = pd.to_numeric(
    freight["truck_id"],
    errors="coerce"
).astype("Int64")

In [ ]:
freight_numeric_columns = [
    "net_revenue",
    "weight_kg",
    "weight_cubic",
    "goods_value"
]

for column in freight_numeric_columns:

    freight[column] = pd.to_numeric(
        freight[column],
        errors="coerce"
    )

freight.dtypes

In [ ]:
invalid_cost_drivers = ~costs["driver_id"].isin(
    drivers_raw["Driver ID"]
)

print(
    "Invalid Driver IDs in Costs:",
    invalid_cost_drivers.sum()
)

In [ ]:
# If invalid records exist:

display(
    costs.loc[invalid_cost_drivers]
)

In [ ]:
invalid_cost_trucks = ~costs["truck_id"].isin(
    vehicles_raw["Truck ID"]
)

print(
    "Invalid Truck IDs in Costs:",
    invalid_cost_trucks.sum()
)

In [ ]:
invalid_freight_trucks = ~freight["truck_id"].isin(
    vehicles_raw["Truck ID"]
)

print(
    "Invalid Truck IDs in Freight:",
    invalid_freight_trucks.sum()
)

In [ ]:
invalid_freight_customers = ~freight["customer_id"].isin(
    customers_raw["Customer ID"]
)

print(
    "Invalid Customer IDs in Freight:",
    invalid_freight_customers.sum()
)

In [ ]:
print(
    "Drivers PK valid:",
    drivers_raw["Driver ID"].notna().all()
    and drivers_raw["Driver ID"].is_unique
)

print(
    "Vehicles PK valid:",
    vehicles_raw["Truck ID"].notna().all()
    and vehicles_raw["Truck ID"].is_unique
)

print(
    "Customers PK valid:",
    customers_raw["Customer ID"].notna().all()
    and customers_raw["Customer ID"].is_unique
)

In [ ]:
def clean_column_names(df):
    """
    Standardizes column names so that they are easier to use
    consistently in Python, PostgreSQL and SQL queries.
    """

    df = df.copy()

    df.columns = (
        df.columns
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r"[()?\s\-\/]+", "_", regex=True)
        .str.replace(r"_+", "_", regex=True)
        .str.strip("_")
    )

    return df

In [ ]:
drivers   = clean_column_names(drivers_raw)
vehicles  = clean_column_names(vehicles_raw)
customers = clean_column_names(customers_raw)
costs     = clean_column_names(costs)
freight   = clean_column_names(freight)

print(drivers.columns.tolist())
print(vehicles.columns.tolist())
print(customers.columns.tolist())
print(costs.columns.tolist())
print(freight.columns.tolist())

In [ ]:
def clean_text_columns(df):
    """
    Removes leading/trailing whitespace and converts empty strings
    into missing values.
    """

    df = df.copy()

    text_columns = df.select_dtypes(
        include=["object", "string"]
    ).columns

    for column in text_columns:

        df[column] = (
            df[column]
            .astype("string")
            .str.strip()
        )

        # Treat completely empty strings as missing values.
        df[column] = df[column].replace(
            r"^\s*$",
            pd.NA,
            regex=True
        )

    return df

In [ ]:
drivers   = clean_text_columns(drivers)
vehicles  = clean_text_columns(vehicles)
customers = clean_text_columns(customers)
costs     = clean_text_columns(costs)
freight   = clean_text_columns(freight)

In [ ]:
numeric_columns = {
    "costs": [
        "km_traveled",
        "liters",
        "fuel",
        "maintenance",
        "fixed_costs"
    ],

    "freight": [
        "net_revenue",
        "weight_kg",
        "weight_cubic",
        "goods_value"
    ]
}

for column in numeric_columns["costs"]:
    costs[column] = pd.to_numeric(
        costs[column],
        errors="coerce"
    )

for column in numeric_columns["freight"]:
    freight[column] = pd.to_numeric(
        freight[column],
        errors="coerce"
    )

In [ ]:
costs["date"] = pd.to_datetime(
    costs["date"],
    errors="coerce"
)

freight["date"] = pd.to_datetime(
    freight["date"],
    errors="coerce"
)

print("Invalid Costs dates:",
      costs["date"].isna().sum())

print("Invalid Freight dates:",
      freight["date"].isna().sum())

In [ ]:
for name, df in {
    "Drivers":   drivers,
    "Vehicles":  vehicles,
    "Customers": customers,
    "Costs":     costs,
    "Freight":   freight
}.items():

    print("\n", name)
    print(df.isna().sum())

In [ ]:
for name, df in {
    "Drivers":   drivers,
    "Vehicles":  vehicles,
    "Customers": customers,
    "Costs":     costs,
    "Freight":   freight
}.items():

    print(
        name,
        "duplicate rows:",
        df.duplicated().sum()
    )

In [ ]:
print(
    "Driver ID duplicates:",
    drivers["driver_id"].duplicated().sum()
)

print(
    "Truck ID duplicates:",
    vehicles["truck_id"].duplicated().sum()
)

print(
    "Customer ID duplicates:",
    customers["customer_id"].duplicated().sum()
)

In [ ]:
print(
    "Negative KM:",
    (costs["km_traveled"] < 0).sum()
)

print(
    "Negative Liters:",
    (costs["liters"] < 0).sum()
)

print(
    "Negative Fuel Cost:",
    (costs["fuel"] < 0).sum()
)

print(
    "Negative Maintenance:",
    (costs["maintenance"] < 0).sum()
)

print(
    "Negative Fixed Cost:",
    (costs["fixed_costs"] < 0).sum()
)

In [ ]:
print(
    "Negative Revenue:",
    (freight["net_revenue"] < 0).sum()
)

print(
    "Negative Weight:",
    (
        (freight["weight_kg"] < 0) |
        (freight["weight_cubic"] < 0)
    ).sum()
)

In [ ]:
invalid_latitude = customers[
    (customers["latitude"] < -90) |
    (customers["latitude"] > 90)
]

invalid_longitude = customers[
    (customers["longitude"] < -180) |
    (customers["longitude"] > 180)
]

print(
    "Invalid latitude:",
    len(invalid_latitude)
)

print(
    "Invalid longitude:",
    len(invalid_longitude)
)

In [ ]:
costs["km_per_liter_check"] = np.where(
    costs["liters"] > 0,
    costs["km_traveled"] / costs["liters"],
    np.nan
)

costs["km_per_liter_check"].describe()

In [ ]:
vehicles["year"] = pd.to_numeric(
    vehicles["year"],
    errors="coerce"
)

vehicles["year"].describe()